In [ ]:
# ============================================================
# ANN Regression Workflow for Stored and Leaked CO2 Mass
# ============================================================
# This script trains Artificial Neural Network regressors to
# predict stored CO2 mass and leaked CO2 mass from CMG-GEM
# simulation-derived input parameters.
#
# Workflow:
# 1. Load processed simulation dataset
# 2. Classify leakage fraction into leakage-risk categories
# 3. Exclude secure cases for leakage-relevant ANN regression
# 4. Apply stratified train/test split
# 5. Apply manual oversampling only to the training set
# 6. Compare ANN architectures and L2 regularization
# 7. Select final ANN configuration
# 8. Train ANN regressors for stored and leaked CO2 mass
# 9. Evaluate ANN models on the unchanged hold-out test set
# 10. Save metrics, parity plots, and error diagnostics
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error


# ============================================================
# USER SETTINGS
# ============================================================

# Change this to your local full dataset path if needed.
DATA_PATH = Path("data/example_dataset.csv")

OUTPUT_DIR = Path(".")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
INNER_VALIDATION_SIZE = 0.20

# Leakage-fraction thresholds.
# Leakage fraction is expressed as a decimal fraction, not percent.
SECURE_MAX = 0.0001      # 0.01%
MID_POINT = 0.01805      # 1.805%
EXTREME_MIN = 0.036      # 3.6%

# Input variables used for ANN regression.
X_COLS = [
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_dist_to_well_in_grid",
    "PERM_caprock_perm",
    "porosity",
    "aquifer_porosity",
    "aquifer_permeability",
]

# Target variables.
TARGET_STORED = "mass_CO2_total"
TARGET_LEAKED = "mass_CO2_leakage"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def classify_leakage_fraction(frac: float) -> str:
    """
    Classify leakage fraction into four leakage-risk categories.

    Parameters
    ----------
    frac : float
        Leakage fraction expressed as decimal fraction, not percent.

    Returns
    -------
    str
        One of: secure, low_risk, high_risk, extreme.
    """
    if pd.isna(frac):
        return np.nan

    if frac <= SECURE_MAX:
        return "secure"
    elif frac <= MID_POINT:
        return "low_risk"
    elif frac <= EXTREME_MIN:
        return "high_risk"
    else:
        return "extreme"


def manual_oversample_low_high(
    train_df: pd.DataFrame,
    label_col: str = "leak_category",
    random_state: int = 42
) -> pd.DataFrame:
    """
    Apply manual oversampling only to the training set.

    Low- and high-leakage cases are duplicated once, while
    extreme-leakage cases are kept unchanged. The resulting
    training set is shuffled using a fixed random seed.
    """
    extreme_df = train_df[train_df[label_col] == "extreme"]
    low_df = train_df[train_df[label_col] == "low_risk"]
    high_df = train_df[train_df[label_col] == "high_risk"]

    low_oversampled = pd.concat([low_df, low_df], ignore_index=True)
    high_oversampled = pd.concat([high_df, high_df], ignore_index=True)

    train_oversampled = pd.concat(
        [extreme_df, low_oversampled, high_oversampled],
        ignore_index=True
    )

    train_oversampled = train_oversampled.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    return train_oversampled


def check_required_columns(df: pd.DataFrame, required_cols: list) -> None:
    """
    Check that all required columns exist in the dataframe.
    """
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            "The following required columns are missing from the dataset: "
            f"{missing_cols}"
        )


def print_class_distribution(df: pd.DataFrame, label_col: str, title: str) -> None:
    """
    Print class counts and proportions.
    """
    print(f"\n=== {title} ===")
    print("\nCounts:")
    print(df[label_col].value_counts(dropna=False))
    print("\nProportions:")
    print(df[label_col].value_counts(normalize=True, dropna=False))


def make_short_label(layers: str, alpha: float) -> str:
    """
    Create short x-axis labels for ANN architecture plots.
    """
    return f"{layers}\nα={alpha:g}"


def train_mlp_regressor(
    X_train_s: np.ndarray,
    y_train: np.ndarray,
    hidden_layers: tuple,
    alpha: float,
    random_state: int = 42
) -> MLPRegressor:
    """
    Train an MLPRegressor with the selected architecture.
    """
    model = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.001,
        max_iter=2000,
        random_state=random_state
    )

    model.fit(X_train_s, y_train)
    return model


def evaluate_regression(
    y_train_true: np.ndarray,
    y_train_pred: np.ndarray,
    y_test_true: np.ndarray,
    y_test_pred: np.ndarray,
    y_baseline: np.ndarray
) -> dict:
    """
    Compute R2 and MAE for train, test, and naive baseline predictions.
    """
    return {
        "R2_train": r2_score(y_train_true, y_train_pred),
        "R2_test": r2_score(y_test_true, y_test_pred),
        "MAE_train_t": mean_absolute_error(y_train_true, y_train_pred),
        "MAE_test_t": mean_absolute_error(y_test_true, y_test_pred),
        "Baseline_R2": r2_score(y_test_true, y_baseline),
        "Baseline_MAE_t": mean_absolute_error(y_test_true, y_baseline),
    }


def save_metrics_table(metrics: dict, target_name: str, file_prefix: str) -> pd.DataFrame:
    """
    Save exact and rounded regression metrics.
    """
    metrics_df = pd.DataFrame({
        "Target": [target_name],
        **{key: [value] for key, value in metrics.items()}
    })

    metrics_display = metrics_df.copy()

    r2_cols = ["R2_train", "R2_test", "Baseline_R2"]
    mae_cols = ["MAE_train_t", "MAE_test_t", "Baseline_MAE_t"]

    metrics_display[r2_cols] = metrics_display[r2_cols].round(4)
    metrics_display[mae_cols] = metrics_display[mae_cols].round(2)

    metrics_df.to_csv(
        TABLES_DIR / f"{file_prefix}_metrics_exact.csv",
        index=False
    )

    metrics_display.to_csv(
        TABLES_DIR / f"{file_prefix}_metrics_rounded.csv",
        index=False
    )

    print(f"\n=== ANN REGRESSION METRICS — {target_name.upper()} ===")
    print(metrics_display.to_string(index=False))

    return metrics_df


def plot_parity(
    y_train_true: np.ndarray,
    y_train_pred: np.ndarray,
    y_test_true: np.ndarray,
    y_test_pred: np.ndarray,
    r2_train: float,
    r2_test: float,
    target_label: str,
    output_path: Path
) -> None:
    """
    Generate parity plot for train and unchanged hold-out test predictions.
    """
    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 13
    })

    fig, ax = plt.subplots(figsize=(6, 6), dpi=600)

    ax.scatter(
        y_train_true,
        y_train_pred,
        color="#4A76FF",
        edgecolor="black",
        linewidth=0.4,
        s=28,
        alpha=0.65,
        label=f"Train ($R^2$ = {r2_train:.4f})"
    )

    ax.scatter(
        y_test_true,
        y_test_pred,
        color="#FF6A6A",
        edgecolor="black",
        linewidth=0.4,
        s=30,
        alpha=0.80,
        label=f"Test ($R^2$ = {r2_test:.4f})"
    )

    lo = min(
        y_train_true.min(),
        y_test_true.min(),
        y_train_pred.min(),
        y_test_pred.min()
    )

    hi = max(
        y_train_true.max(),
        y_test_true.max(),
        y_train_pred.max(),
        y_test_pred.max()
    )

    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(f"True {target_label} (t)", fontsize=16)
    ax.set_ylabel(f"Predicted {target_label} (t)", fontsize=16)
    ax.set_title(f"Parity Plot — {target_label.title()}", fontsize=18)

    ax.tick_params(axis="both", labelsize=14)

    legend = ax.legend(
        loc="lower right",
        frameon=True,
        fancybox=True,
        framealpha=0.90,
        edgecolor="black",
        fontsize=13
    )

    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_facecolor("white")

    ax.grid(alpha=0.25)

    plt.tight_layout()
    fig.savefig(output_path, dpi=600, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved parity plot: {output_path}")


def plot_absolute_error_distribution(
    y_test_true: np.ndarray,
    y_test_pred: np.ndarray,
    target_label: str,
    output_path: Path
) -> None:
    """
    Generate absolute error distribution for unchanged hold-out test set.
    """
    abs_error = np.abs(y_test_pred - y_test_true)

    fig, ax = plt.subplots(figsize=(6.2, 4.4), dpi=600)

    ax.hist(
        abs_error,
        bins=35,
        edgecolor="black",
        linewidth=0.4,
        alpha=0.85
    )

    ax.set_xlabel(f"Absolute error in {target_label} (t)", fontsize=13)
    ax.set_ylabel("Frequency", fontsize=13)
    ax.set_title(f"Absolute Error Distribution — {target_label.title()}", fontsize=14)

    ax.tick_params(axis="both", labelsize=11)
    ax.grid(axis="y", alpha=0.25, linestyle="--")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    fig.savefig(output_path, dpi=600, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved error distribution: {output_path}")


# ============================================================
# LOAD DATASET
# ============================================================

print("\nLoading dataset...")
df = pd.read_csv(DATA_PATH)

print("Dataset loaded:", df.shape)
print(df.head())

required_cols = X_COLS + ["leakage_fraction", TARGET_STORED, TARGET_LEAKED]
check_required_columns(df, required_cols)


# ============================================================
# LEAKAGE-RISK CLASSIFICATION
# ============================================================

df["leak_category"] = df["leakage_fraction"].apply(classify_leakage_fraction)

print_class_distribution(
    df=df,
    label_col="leak_category",
    title="Full dataset leakage-category distribution"
)


# ============================================================
# ANN DATASET AND TRAIN / TEST SPLIT
# ============================================================

# Exclude secure cases for leakage-relevant ANN regression.
df_ann = df[df["leak_category"] != "secure"].copy()
df_ann.reset_index(drop=True, inplace=True)

print_class_distribution(
    df=df_ann,
    label_col="leak_category",
    title="Leakage-relevant ANN dataset distribution"
)

train_df_ann, test_df_ann = train_test_split(
    df_ann,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_ann["leak_category"]
)

train_df_ann_os = manual_oversample_low_high(
    train_df=train_df_ann,
    label_col="leak_category",
    random_state=RANDOM_STATE
)

print_class_distribution(
    df=train_df_ann,
    label_col="leak_category",
    title="ANN training-set distribution before oversampling"
)

print_class_distribution(
    df=train_df_ann_os,
    label_col="leak_category",
    title="ANN training-set distribution after oversampling"
)

print_class_distribution(
    df=test_df_ann,
    label_col="leak_category",
    title="ANN unchanged hold-out test-set distribution"
)

ann_distribution = pd.DataFrame({
    "original_train_count": train_df_ann["leak_category"].value_counts(),
    "oversampled_train_count": train_df_ann_os["leak_category"].value_counts(),
    "test_count": test_df_ann["leak_category"].value_counts()
})

ann_distribution.to_csv(
    TABLES_DIR / "ANN_Train_Test_Class_Distribution.csv",
    index=True
)


# ============================================================
# INTERNAL TRAIN / VALIDATION SPLIT FOR ARCHITECTURE SELECTION
# ============================================================
# Architecture comparison is performed on the oversampled training set.
# Stored CO2 mass is used as the tuning target because it provides a
# stable mass-based regression target for comparing hidden-layer and
# L2 regularization effects.

train_inner_df, val_inner_df = train_test_split(
    train_df_ann_os,
    test_size=INNER_VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_df_ann_os["leak_category"]
)

X_train_inner = train_inner_df[X_COLS].values
y_train_inner = train_inner_df[TARGET_STORED].values

X_val_inner = val_inner_df[X_COLS].values
y_val_inner = val_inner_df[TARGET_STORED].values

scaler_X_inner = StandardScaler()
X_train_inner_s = scaler_X_inner.fit_transform(X_train_inner)
X_val_inner_s = scaler_X_inner.transform(X_val_inner)


# ============================================================
# COMPARE ANN ARCHITECTURES AND L2 REGULARIZATION
# ============================================================

architecture_grid = [
    ((32,), 0.0),
    ((64, 32), 0.0),
    ((128, 64, 32), 0.0),
    ((64, 32), 1e-4),
    ((64, 32), 1e-3),
    ((128, 64, 32), 1e-4),
]

results = []

for hidden_layers, alpha in architecture_grid:
    model = train_mlp_regressor(
        X_train_s=X_train_inner_s,
        y_train=y_train_inner,
        hidden_layers=hidden_layers,
        alpha=alpha,
        random_state=RANDOM_STATE
    )

    y_train_inner_pred = model.predict(X_train_inner_s)
    y_val_inner_pred = model.predict(X_val_inner_s)

    results.append({
        "Layers": str(hidden_layers),
        "Alpha_L2": alpha,
        "R2_train": r2_score(y_train_inner, y_train_inner_pred),
        "R2_validation": r2_score(y_val_inner, y_val_inner_pred),
        "MAE_train_t": mean_absolute_error(y_train_inner, y_train_inner_pred),
        "MAE_validation_t": mean_absolute_error(y_val_inner, y_val_inner_pred)
    })

df_arch_all = pd.DataFrame(results)
df_arch_all["Underfitting_flag"] = df_arch_all["R2_validation"] < 0

df_arch_all_display = df_arch_all.copy()
df_arch_all_display[["R2_train", "R2_validation"]] = (
    df_arch_all_display[["R2_train", "R2_validation"]].round(4)
)
df_arch_all_display[["MAE_train_t", "MAE_validation_t"]] = (
    df_arch_all_display[["MAE_train_t", "MAE_validation_t"]].round(2)
)

print("\n=== ANN ARCHITECTURE COMPARISON — FULL RESULTS ===")
print(df_arch_all_display.to_string(index=False))

df_arch_all.to_csv(
    TABLES_DIR / "Table_7_ANN_ArchitectureComparison_full_exact.csv",
    index=False
)

df_arch_all_display.to_csv(
    TABLES_DIR / "Table_7_ANN_ArchitectureComparison_full_rounded.csv",
    index=False
)

# Exclude clearly failed underfitting cases from paper plot/table.
df_arch_plot = df_arch_all[df_arch_all["R2_validation"] >= 0].copy().reset_index(drop=True)

df_arch_plot_display = df_arch_plot.copy()
df_arch_plot_display[["R2_train", "R2_validation"]] = (
    df_arch_plot_display[["R2_train", "R2_validation"]].round(4)
)
df_arch_plot_display[["MAE_train_t", "MAE_validation_t"]] = (
    df_arch_plot_display[["MAE_train_t", "MAE_validation_t"]].round(2)
)

print("\n=== ANN ARCHITECTURE COMPARISON — FILTERED FOR PAPER ===")
print(df_arch_plot_display.to_string(index=False))

df_arch_plot.to_csv(
    TABLES_DIR / "Table_7_ANN_ArchitectureComparison_filtered_exact.csv",
    index=False
)

df_arch_plot_display.to_csv(
    TABLES_DIR / "Table_7_ANN_ArchitectureComparison_filtered_rounded.csv",
    index=False
)


# ============================================================
# SELECT FINAL ANN CONFIGURATION
# ============================================================
# Selection rule:
# Select the best validation R2, while preferring L2 regularization
# when validation performance is effectively equivalent.

R2_TOL = 1e-4

best_r2 = df_arch_plot["R2_validation"].max()

candidate_models = df_arch_plot[
    df_arch_plot["R2_validation"] >= best_r2 - R2_TOL
].copy()

candidate_regularized = candidate_models[candidate_models["Alpha_L2"] > 0]

if not candidate_regularized.empty:
    selected_row = (
        candidate_regularized
        .sort_values(by=["R2_validation", "MAE_validation_t"], ascending=[False, True])
        .iloc[0]
    )
else:
    selected_row = (
        candidate_models
        .sort_values(by=["R2_validation", "MAE_validation_t"], ascending=[False, True])
        .iloc[0]
    )

FINAL_HIDDEN_LAYERS = eval(selected_row["Layers"])
FINAL_ALPHA = selected_row["Alpha_L2"]

print("\n=== SELECTED FINAL ANN CONFIGURATION ===")
print("Selection rule: best validation R2, preferring L2 regularization when performance is equivalent.")
print("Hidden layers:", FINAL_HIDDEN_LAYERS)
print("Alpha:", FINAL_ALPHA)
print("Validation R2:", selected_row["R2_validation"])
print("Validation MAE:", selected_row["MAE_validation_t"])


# ============================================================
# PLOT ARCHITECTURE COMPARISON
# ============================================================

plot_labels = [
    make_short_label(row["Layers"], row["Alpha_L2"])
    for _, row in df_arch_plot.iterrows()
]

selected_mask = (
    (df_arch_plot["Layers"] == str(FINAL_HIDDEN_LAYERS)) &
    (df_arch_plot["Alpha_L2"] == FINAL_ALPHA)
)

selected_pos = np.where(selected_mask.values)[0][0]
x = np.arange(len(df_arch_plot))

y_min = min(
    df_arch_plot["R2_train"].min(),
    df_arch_plot["R2_validation"].min()
)

y_max = max(
    df_arch_plot["R2_train"].max(),
    df_arch_plot["R2_validation"].max()
)

margin = max(0.00005, (y_max - y_min) * 0.30)

ylim_low = y_min - margin
ylim_high = min(1.00005, y_max + margin)

fig, ax = plt.subplots(figsize=(8.8, 5.4), dpi=600)

ax.plot(
    x,
    df_arch_plot["R2_train"],
    color="royalblue",
    marker="s",
    markersize=7,
    linewidth=2.2,
    label="Train $R^2$"
)

ax.plot(
    x,
    df_arch_plot["R2_validation"],
    color="crimson",
    marker="o",
    markersize=7,
    linewidth=2.2,
    label="Validation $R^2$"
)

ax.scatter(
    selected_pos,
    df_arch_plot.iloc[selected_pos]["R2_validation"],
    s=90,
    color="crimson",
    edgecolor="black",
    linewidth=0.9,
    zorder=5
)

ax.annotate(
    "Selected",
    xy=(selected_pos, df_arch_plot.iloc[selected_pos]["R2_validation"]),
    xytext=(0, 14),
    textcoords="offset points",
    ha="center",
    fontsize=11,
    fontweight="bold",
    color="black"
)

for i, val in enumerate(df_arch_plot["R2_validation"]):
    ax.annotate(
        f"{val:.4f}",
        xy=(i, val),
        xytext=(0, -18),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        color="crimson"
    )

ax.set_xticks(x)
ax.set_xticklabels(plot_labels, rotation=0, ha="center", fontsize=11)

ax.set_ylabel("$R^2$ score", fontsize=14)
ax.set_xlabel("Model configuration", fontsize=14)
ax.set_title("Effect of Hidden Layers and L2 Regularization", fontsize=18, fontweight="bold")

ax.set_ylim(ylim_low, ylim_high)

ax.grid(True, alpha=0.25, linestyle="--")
ax.legend(frameon=False, loc="lower right", fontsize=12)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.text(
    0.5,
    -0.02,
    "The single-hidden-layer configuration (32,) was excluded from the main plot due to severe underfitting (negative validation $R^2$).",
    ha="center",
    fontsize=10
)

plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_7_ANN_ArchitectureComparison.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"Saved architecture comparison figure: {fig_path}")


# ============================================================
# TRAIN ANN REGRESSOR — STORED CO2 MASS
# ============================================================

X_total_train = train_df_ann_os[X_COLS].values
y_total_train = train_df_ann_os[TARGET_STORED].values

X_total_test = test_df_ann[X_COLS].values
y_total_test = test_df_ann[TARGET_STORED].values

scaler_X_total = StandardScaler()
X_total_train_s = scaler_X_total.fit_transform(X_total_train)
X_total_test_s = scaler_X_total.transform(X_total_test)

mlp_total = train_mlp_regressor(
    X_train_s=X_total_train_s,
    y_train=y_total_train,
    hidden_layers=FINAL_HIDDEN_LAYERS,
    alpha=FINAL_ALPHA,
    random_state=RANDOM_STATE
)

y_total_pred_train = mlp_total.predict(X_total_train_s)
y_total_pred_test = mlp_total.predict(X_total_test_s)

y_total_baseline = np.full_like(
    y_total_test,
    fill_value=np.mean(y_total_train),
    dtype=float
)

stored_metrics = evaluate_regression(
    y_train_true=y_total_train,
    y_train_pred=y_total_pred_train,
    y_test_true=y_total_test,
    y_test_pred=y_total_pred_test,
    y_baseline=y_total_baseline
)

save_metrics_table(
    metrics=stored_metrics,
    target_name="Stored CO2 mass",
    file_prefix="ANN_StoredCO2Mass"
)

plot_parity(
    y_train_true=y_total_train,
    y_train_pred=y_total_pred_train,
    y_test_true=y_total_test,
    y_test_pred=y_total_pred_test,
    r2_train=stored_metrics["R2_train"],
    r2_test=stored_metrics["R2_test"],
    target_label="stored CO₂ mass",
    output_path=FIGURES_DIR / "Figure_8a_ANN_Parity_StoredCO2Mass.png"
)


# ============================================================
# TRAIN ANN REGRESSOR — LEAKED CO2 MASS
# ============================================================

X_leak_train = train_df_ann_os[X_COLS].values
y_leak_train = train_df_ann_os[TARGET_LEAKED].values

X_leak_test = test_df_ann[X_COLS].values
y_leak_test = test_df_ann[TARGET_LEAKED].values

scaler_X_leak = StandardScaler()
X_leak_train_s = scaler_X_leak.fit_transform(X_leak_train)
X_leak_test_s = scaler_X_leak.transform(X_leak_test)

# Standardize leaked CO2 target during training.
scaler_y_leak = StandardScaler()
y_leak_train_s = scaler_y_leak.fit_transform(
    y_leak_train.reshape(-1, 1)
).ravel()

mlp_leak = train_mlp_regressor(
    X_train_s=X_leak_train_s,
    y_train=y_leak_train_s,
    hidden_layers=FINAL_HIDDEN_LAYERS,
    alpha=FINAL_ALPHA,
    random_state=RANDOM_STATE
)

y_leak_pred_train_s = mlp_leak.predict(X_leak_train_s)
y_leak_pred_test_s = mlp_leak.predict(X_leak_test_s)

y_leak_pred_train = scaler_y_leak.inverse_transform(
    y_leak_pred_train_s.reshape(-1, 1)
).ravel()

y_leak_pred_test = scaler_y_leak.inverse_transform(
    y_leak_pred_test_s.reshape(-1, 1)
).ravel()

# Avoid physically meaningless negative leaked-mass predictions in diagnostics.
y_leak_pred_train = np.maximum(y_leak_pred_train, 0.0)
y_leak_pred_test = np.maximum(y_leak_pred_test, 0.0)

y_leak_baseline = np.full_like(
    y_leak_test,
    fill_value=np.mean(y_leak_train),
    dtype=float
)

leak_metrics = evaluate_regression(
    y_train_true=y_leak_train,
    y_train_pred=y_leak_pred_train,
    y_test_true=y_leak_test,
    y_test_pred=y_leak_pred_test,
    y_baseline=y_leak_baseline
)

save_metrics_table(
    metrics=leak_metrics,
    target_name="Leaked CO2 mass",
    file_prefix="ANN_LeakedCO2Mass"
)

plot_parity(
    y_train_true=y_leak_train,
    y_train_pred=y_leak_pred_train,
    y_test_true=y_leak_test,
    y_test_pred=y_leak_pred_test,
    r2_train=leak_metrics["R2_train"],
    r2_test=leak_metrics["R2_test"],
    target_label="leaked CO₂ mass",
    output_path=FIGURES_DIR / "Figure_8b_ANN_Parity_LeakedCO2Mass.png"
)


# ============================================================
# COMBINED PERFORMANCE TABLE
# ============================================================

combined_metrics = pd.concat([
    pd.DataFrame({"Target": ["Stored CO2 mass"], **{k: [v] for k, v in stored_metrics.items()}}),
    pd.DataFrame({"Target": ["Leaked CO2 mass"], **{k: [v] for k, v in leak_metrics.items()}})
], ignore_index=True)

combined_metrics_display = combined_metrics.copy()

r2_cols = ["R2_train", "R2_test", "Baseline_R2"]
mae_cols = ["MAE_train_t", "MAE_test_t", "Baseline_MAE_t"]

combined_metrics_display[r2_cols] = combined_metrics_display[r2_cols].round(4)
combined_metrics_display[mae_cols] = combined_metrics_display[mae_cols].round(2)

combined_metrics.to_csv(
    TABLES_DIR / "ANN_RegressionMetrics_combined_exact.csv",
    index=False
)

combined_metrics_display.to_csv(
    TABLES_DIR / "ANN_RegressionMetrics_combined_rounded.csv",
    index=False
)

print("\n=== COMBINED ANN REGRESSION METRICS ===")
print(combined_metrics_display.to_string(index=False))


# ============================================================
# SAVE MODEL INPUT INFORMATION
# ============================================================

input_info = pd.DataFrame({
    "input_variable": X_COLS
})

input_info.to_csv(
    TABLES_DIR / "ANN_InputVariables.csv",
    index=False
)

print("\nSaved ANN workflow outputs in:")
print(f"- Figures: {FIGURES_DIR.resolve()}")
print(f"- Tables:  {TABLES_DIR.resolve()}")

print("\nANN regression workflow completed successfully.")